In [1]:
import pandas as pd

df = pd.read_csv("../../../data/bank-marketing/bank-full.csv", sep=";")
print(df.head())

print(df.shape)

assert "y" in df.columns

df.dtypes

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  
(45211, 17)


age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [2]:
#Converts "yes" to 1 & "no" to 0
assert df.y.isin(["yes", "no"]).all()
print(df.y.unique())
mapping = {"yes": 1, "no": 0}
y = df["y"].map(mapping)

#Displays number of observations in each class
print(y.value_counts())

#Displays the percentage of observations in each class
print(y.value_counts(normalize=True) * 100)

#Confirms there are no missing values
assert y.notna().all(axis=None)

<StringArray>
['no', 'yes']
Length: 2, dtype: str
y
0    39922
1     5289
Name: count, dtype: int64
y
0    88.30152
1    11.69848
Name: proportion, dtype: float64


In [3]:
X = df.drop(columns=["y", "duration"])

assert "y" not in X.columns and "duration" not in X.columns

print(X.shape)
print(X.columns)

(45211, 15)
Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous',
       'poutcome'],
      dtype='str')


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)



In [5]:
print(X_train.shape, X_temp.shape, y_train.shape, y_temp.shape)

print(y_train.value_counts(normalize=True))

print(y_temp.value_counts(normalize=True))

assert X_train.shape[0] == y_train.shape[0]
assert X_temp.shape[0] == y_temp.shape[0]

assert X_train.shape[0] + X_temp.shape[0] == df.shape[0]

(27126, 15) (18085, 15) (27126,) (18085,)
y
0    0.883027
1    0.116973
Name: proportion, dtype: float64
y
0    0.882997
1    0.117003
Name: proportion, dtype: float64


In [6]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.value_counts(normalize=True), y_val.value_counts(normalize=True), y_test.value_counts(normalize=True))

(27126, 15) (9042, 15) (9043, 15)
y
0    0.883027
1    0.116973
Name: proportion, dtype: float64 y
0    0.88299
1    0.11701
Name: proportion, dtype: float64 y
0    0.883003
1    0.116997
Name: proportion, dtype: float64


In [7]:
import numpy as np
from sklearn.metrics import accuracy_score

majority_class = y_train.value_counts().idxmax()
print(f"Majority Class: {majority_class}")
assert majority_class in [0, 1]

y_val_pred = np.full(X_val.shape[0], majority_class)
print(y_val_pred[:10])

assert len(y_val_pred) == len(y_val)

assert np.all(y_val_pred == majority_class)

baseline_accuracy = round(accuracy_score(y_val, y_val_pred) * 100, 2)
print(f"value:{baseline_accuracy}%")

calculated_accuracy = round((y_val == 0).sum() / len(y_val) * 100, 2)

assert np.isclose(baseline_accuracy, calculated_accuracy)

Majority Class: 0
[0 0 0 0 0 0 0 0 0 0]
value:88.3%


In [8]:
assert len(X_train.index.intersection(X_val.index)) == 0
assert len(X_test.index.intersection(X_val.index)) == 0
assert len(X_train.index.intersection(X_test.index)) == 0

## Day 15 — Bank Subscription Classification Results

### 1. Problem Definition

The objective is to develop a binary classification system that predicts whether a bank customer will subscribe to a term deposit following a marketing campaign.

Predictions are intended to occur before the current marketing phone call.

Target encoding:
- `0`: Customer did not subscribe.
- `1`: Customer subscribed.

The eventual goal is to identify customers who are more likely to subscribe, helping the bank prioritize marketing efforts.

### 2. Dataset Overview

**Source:** UCI Bank Marketing Dataset (`bank-full.csv`)

- Total observations: 45,211
- Original columns: 17
- Input features after exclusions: 15
- Target: `y`
- File delimiter: Semicolon (`;`)

**Target distribution:**

| Class | Observations | Percentage |
|---|---:|---:|
| 0 — No subscription | 39,922 | 88.30% |
| 1 — Subscription | 5,289 | 11.70% |

The dataset is imbalanced, with significantly fewer positive observations than negative observations.

### 3. Data Preparation and Leakage Prevention

The target was validated against the expected categories (`yes`, `no`) and encoded using an explicit mapping dictionary.

Two columns were excluded from the feature matrix:

- `y`: The prediction target.
- `duration`: The duration of the current marketing call, which is unavailable before the call occurs.

The original DataFrame was preserved.

Assertions verified the target encoding, feature exclusions, and absence of missing encoded labels.

### 4. Stratified Dataset Splitting

Used scikit-learn's `train_test_split()` with `random_state=42` and stratification.

Splitting occurred in two stages:

1. Original dataset → 60% training and 40% temporary.
2. Temporary dataset → 50% validation and 50% test.

**Final partitions:**

| Partition | Observations | Percentage |
|---|---:|---:|
| Training | 27,126 | 60% |
| Validation | 9,042 | 20% |
| Test | 9,043 | 20% |
| Total | 45,211 | 100% |

All three partitions preserved approximately 88.3% non-subscribers and 11.7% subscribers.

Assertions verified:
- Matching feature and target lengths.
- Total observation counts.
- No original row-index overlap across partitions.

**Limitation:** Random stratification does not establish chronological independence. Future experiments should consider time-based splitting to simulate prediction on later marketing campaigns. Index-disjoint partitions also do not guarantee customer-level independence or the absence of duplicate records.

### 5. Majority-Class Baseline

The majority class was determined exclusively from the training target using `value_counts().idxmax()`.

**Majority class:** `0` (No subscription)

A NumPy array was generated to predict the majority class for every validation observation.

Validation accuracy was calculated using scikit-learn's `accuracy_score()` and independently checked against the proportion of non-subscribers in validation.

**Validation accuracy: 88.30%**

No machine learning estimator was trained during this session.

### 6. Baseline Limitations

Although the baseline achieves 88.30% accuracy, it never predicts a positive subscription.

Consequently:
- It identifies zero actual subscribers.
- It has zero recall for the positive class.
- Accuracy alone is insufficient to evaluate its business usefulness.

Future classifiers should be compared using metrics such as precision, recall, F1-score, and ROC AUC, alongside accuracy.

### 7. Engineering Verification

Completed checks included:

- Dataset parsing and column verification.
- Binary target validation.
- Feature exclusion assertions.
- Stratified class-distribution inspection.
- Partition size and alignment assertions.
- Pairwise row-index overlap checks.
- Prediction-length and prediction-value assertions.
- A mutation test confirming that incorrect predictions are detected.
- Independent verification of baseline accuracy.

The test partition was reserved for later model evaluation. Its class distribution was inspected, but no test-set performance metric was calculated or used for model selection.

### 8. Conclusion

Day 15 established the foundation for a reproducible bank subscription classification workflow.

The central finding is that class imbalance can produce deceptively high classification accuracy. The 88.30% majority-class baseline provides a reference against which future classification models can be evaluated.

**Next:** Develop a classification model that can identify positive subscriptions and evaluate its performance beyond accuracy.

## Day 16 — Mixed-Feature Preprocessing

In [9]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()
print(numeric_features)
print(len(numeric_features))
print(categorical_features)
print(len(categorical_features))

assert set(numeric_features).isdisjoint(set(categorical_features))
assert set(numeric_features) | set(categorical_features) == set(X_train.columns)

assert "duration" not in numeric_features and "y" not in numeric_features
assert "duration" not in categorical_features and "y" not in categorical_features

assert (len(numeric_features) + len(categorical_features)) == 15

['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
6
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
9


/tmp/ipykernel_57398/238417453.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include="object").columns.tolist()


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessing_pipeline = Pipeline(
    [("preprocessing", preprocessor)]
)

print(type(preprocessing_pipeline.steps))
print(type(numeric_features))
print(numeric_features)

assert isinstance(numeric_features, list)
assert isinstance(categorical_features, list)
assert len(numeric_features) == 6
assert len(categorical_features) == 9

<class 'list'>
<class 'list'>
['age', 'balance', 'day', 'campaign', 'pdays', 'previous']


In [11]:
preprocessing_pipeline.fit(X_train)
X_train_processed = preprocessing_pipeline.transform(X_train)
X_val_processed = preprocessing_pipeline.transform(X_val)

print(X_train_processed.shape)
print(X_val_processed.shape)

(27126, 50)
(9042, 50)


In [12]:
X_val_unknown = X_val.copy()
assert X_val.equals(X_val_unknown)

X_val_unknown.loc[X_val_unknown.index[0], "job"] = "astronaut"
X_val_unknown_processed = preprocessing_pipeline.transform(X_val_unknown)

assert X_val_unknown.loc[X_val_unknown.index[0], "job"] == "astronaut"

assert X_val_unknown_processed.shape == X_val_processed.shape

In [13]:
feature_names = preprocessing_pipeline.get_feature_names_out()
cat_job = [name for name in feature_names if name.startswith("cat__job_")]
print(cat_job)
print(len(cat_job))

['cat__job_admin.', 'cat__job_blue-collar', 'cat__job_entrepreneur', 'cat__job_housemaid', 'cat__job_management', 'cat__job_retired', 'cat__job_self-employed', 'cat__job_services', 'cat__job_student', 'cat__job_technician', 'cat__job_unemployed', 'cat__job_unknown']
12


In [14]:
job_indices = []

for i, name in enumerate(feature_names):
    if feature_names[i].startswith("cat__job_"):
        print(i, name)
        job_indices.append(i)

assert len(job_indices) == 12
print(job_indices)

job_values = X_val_unknown_processed[0, job_indices]

assert np.all(job_values == 0)
# False

6 cat__job_admin.
7 cat__job_blue-collar
8 cat__job_entrepreneur
9 cat__job_housemaid
10 cat__job_management
11 cat__job_retired
12 cat__job_self-employed
13 cat__job_services
14 cat__job_student
15 cat__job_technician
16 cat__job_unemployed
17 cat__job_unknown
[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


## Day 16 — Mixed-Feature Preprocessing Results

### Objective
Build a reusable scikit-learn preprocessing pipeline that handles numerical and categorical features, prevents preprocessing leakage, and supports unseen categories during inference.

### Implementation
- Identified **6 numerical features** and **9 categorical features** using Pandas data types.
- Constructed a `ColumnTransformer` that passes numerical features through unchanged and applies `OneHotEncoder(handle_unknown="ignore")` to categorical features.
- Wrapped the transformer in a scikit-learn `Pipeline`.
- Fitted the pipeline exclusively on `X_train`.
- Transformed both training and validation data using the fitted pipeline.
- Preserved the exclusion of `duration` because it is unavailable before the marketing call.

### Results

| Dataset | Original shape | Transformed shape |
|---|---|---|
| Training | (27,126, 15) | (27,126, 50) |
| Validation | (9,042, 15) | (9,042, 50) |

The 50 transformed features consist of 6 numerical features and 44 one-hot-encoded categorical features.

### Verification
- Confirmed that numerical and categorical feature groups are disjoint and collectively cover all 15 input features.
- Confirmed that feature selectors contain column names rather than DataFrames.
- Confirmed the pipeline produces consistent feature dimensions.
- Tested an unseen `job="astronaut"` category without refitting the pipeline.
- Verified that all 12 encoded job indicators are zero for the unseen category.
- Replaced hardcoded column positions with dynamically identified feature indices.
- Successfully restarted the Jupyter kernel and executed all cells.

### Engineering Lessons
- `.fit()` learns preprocessing parameters; `.transform()` applies them without relearning.
- `handle_unknown="ignore"` prevents errors for unseen categories but does not create new feature columns.
- An unseen category is not automatically mapped to a literal `"unknown"` category learned during training.
- Pandas chained assignment can silently invalidate intended test modifications.
- Passing assertions does not guarantee that a test verifies its intended behavior.
- Jupyter execution order and inconsistent variable names can hide reproducibility defects.

### Limitations and Next Steps
- Numerical features are currently passed through without scaling, imputation, or special handling of sentinel values such as `pdays=-1`.
- No classifier has been trained yet.
- No positive-class precision, recall, F1, or ROC AUC has been measured.
- The final test dataset remains reserved for later evaluation.

**Status:** Day 16 core objectives completed.